<a href="https://colab.research.google.com/github/mdmostafizurrahman6351/LLM_Data/blob/main/LLM_Model/BanglaGPT001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoders


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("sagorsarker/bangla-wikipedia")

In [ ]:
data = ds['train']['content']

print(len(data))
data[:3]

In [ ]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.decoder = ByteLevelDecoders()
trainer = BpeTrainer(
    vocab_size=30000,
    min_frequency=2,
    special_tokens=[
    "[UNK]",
    "[PAD]",
    "[BOS]",
    "[EOS]"
    ],
    initial_alphabet=ByteLevel.alphabet()
)

tokenizer.train_from_iterator(
    data,
    trainer=trainer
)

vocab_size = tokenizer.get_vocab()
print(f"Vocab size: {len(vocab_size)}")

enc = tokenizer.encode("আমি বাংলায় গান গাই।")
print(enc.tokens)
print(enc.ids)
print(tokenizer.decode(enc.ids))


In [ ]:
tokenizer.save('BanglaTokenizer.json')

In [ ]:
tokenizer = Tokenizer.from_file('/content/BanglaTokenizer.json')
enc = tokenizer.encode("আমি বাংলায় গান গাই।")
vocab_size = tokenizer.get_vocab()
print(f"Vocab size: {len(vocab_size)}")
print(enc.tokens)
print(enc.ids)
print(tokenizer.decode(enc.ids))

In [ ]:
filePath = "/content/Bangla_data.bin"

if os.path.exists(filePath):
    os.remove(filePath)

batch_size = 10000

for i in range(0, len(data), batch_size):
    batch = list(data[i: i+batch_size])

    enc_batch = tokenizer.encode_batch(batch)

    batch_ids = []
    for enc in enc_batch:
        batch_ids.extend(enc.ids)

    batch_array = np.array(batch_ids, dtype=np.uint16)

    with open(filePath, 'ab') as f:
        f.write(batch_array.tobytes())



In [ ]:
token_ids = np.memmap('/content/Bangla_data.bin', dtype=np.uint16, mode='r')

In [ ]:
split = int(len(token_ids)*0.9)
train_data = token_ids[:split]
val_data = token_ids[split:]

print(f"Train data size: {len(train_data)}")
print(f"Val data size: {len(val_data)}")


In [ ]:
d_model = 1024
num_heads = 8
block_size = 128
batch_size = 32
num_layers = 16
max_seq_len = 1500
lr = 1e-3
num_epochs = 10


In [ ]:

class LanguageModelDataset(Dataset):
    def __init__(self, token_ids, block_size):
        super().__init__()
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        chunk = self.token_ids[idx: idx+self.block_size+1]

        chunk_tensor = torch.from_numpy(chunk.copy()).to(torch.long)

        x = chunk_tensor[:-1]
        y = chunk_tensor[1:]
        return x, y

train_dataset = LanguageModelDataset(
    train_data,
    block_size
)
val_dataset = LanguageModelDataset(
    val_data,
    block_size
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=True
)

x, y = next(iter(train_dataloader))

print("X:", x.shape)
print(x)
print("Y:", y.shape)
print(y)


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_seq_len,d_model):
        super().__init__()

        pe = torch.zeros(max_seq_len, d_model)

        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1),:]




In [ ]:
class MaskMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)

        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(
            torch.ones(block_size, block_size)
        )
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        Q, K, V = qkv.chunk(3, dim=-1)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        score = (Q @ K.transpose(-1, -2)) / (self.head_dim ** 0.5)
        score = score.masked_fill(self.mask[:T, :T] == 0, float('-inf'))

        score = F.softmax(score, dim=-1)

        out = score @ V
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)

        return out




In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )
    def forward(self, x):
        return self.net(x)


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attention = MaskMultiHeadAttention(d_model, num_heads, block_size)

        self.feedforward = FeedForward(d_model)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.feedforward(self.ln2(x))
        return x

In [ ]:
class GptModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, block_size, num_layers, max_seq_len):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = PositionalEncoding(max_seq_len=max_seq_len, d_model=d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, block_size) for _ in range(num_layers)
        ])

        self.f_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, target=None):
        B, T = x.shape

        x = self.token_embedding(x)
        x = self.position_embedding(x)
        for block in self.blocks:
            x = block(x)
        x = self.f_ln(x)
        logits = self.lm_head(x)

        loss = None
        if target is not None:

            B, T, C = logits.shape
            feature = logits.view(B * T, C)
            target = target.view(B * T)

            loss = F.cross_entropy(feature, target)
        return logits, loss

    @torch.no_grad()
    def genarator(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx



In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size = len(tokenizer.get_vocab())
Model = GptModel(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    block_size=block_size,
    num_layers=num_layers,
    max_seq_len=max_seq_len
).to(device)

In [ ]:
optimizer = torch.optim.AdamW(Model.parameters(), lr=lr)


In [ ]:
for epoch in range(num_epochs):
    Model.train()
    total_loss = 0
    for batch_ids, (x,y) in enumerate(train_dataloader):
        x = x.to(device)
        y = y.to(device)

        logits, loss = Model(x=x, target=y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if (batch_ids % 100 == 0)
            print(f"Epoch: {epoch+1}, Batch: {batch_ids+1}, loss: {loss.item():.4f}")
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch: {epoch+1}, Avg Loss: {avg_loss:.4f}")


In [ ]:
with torch.no_grade():
    Model.evel()

    total_loss = 0
    for batch_ids, (x,y) in enumerate(val_dataloader):
        x = x.to(device)
        y = y.to(device)

        logits, loss = Model(x, y)

        total_loss += loss.item()
    avg_loss = total_loss / len(val_dataloader)
    print(f"Val Loss: {avg_loss:.4f}")


In [ ]:
Model.save('BanglaGPT.h5')

In [ ]:
while(True):
    inp = input('Enter your Question: ')
    if inp == 'exit':
        break
    inp = torch.tensor(tokenizer.encode(inp), dtype=torcj.log).unsqueeze(0).to(device)

    output = Model.genarator(inp,max_seq_len=10)
    output = output.squeeze(0)
    output = tokenizer.decode(output.tolist())
    print(output)
